# Sentiment Classification Project

In [1]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[ 0.2241,  0.0382,  0.0909],
        [-0.0823, -0.6541, -0.2856],
        [-0.5465,  0.4810, -0.9986]], device='cuda:0')


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
import torch
x = torch.randn(10, 10).cuda()
print(x @ x)

tensor([[-4.0342,  3.2047, -2.2161, -2.9718, -3.4613,  5.6683, -4.6254,  2.1467,
          2.2832,  5.1759],
        [ 1.6939,  4.1082,  2.1252, -5.1404,  2.0217,  1.1556,  1.2324,  0.8230,
          1.1620,  1.6949],
        [-1.4679, -2.0262, -0.4036,  1.3218, -1.8469,  2.0694, -0.9350, -1.8322,
          0.2816, -1.2362],
        [ 0.3493, -6.3275,  2.3641,  0.5113,  4.4319, -2.2402,  2.2877, -0.7879,
         -1.2176,  1.5825],
        [ 5.0671,  2.9110,  3.5852, -1.6421,  0.8943,  0.3166, -0.9146,  0.7202,
          1.6263, -2.8202],
        [-3.9229,  0.8291, -1.1639, -0.4340,  2.3665, -2.5017,  0.8992,  2.4516,
         -2.7903,  3.8787],
        [-1.2366, -6.3996,  1.6320,  4.7712, -3.4148,  0.3447, -3.6301, -4.5121,
          0.9828, -4.7936],
        [-6.9976,  6.4944, -4.4692, -1.7790, -4.2447,  3.3477, -1.3347, -4.3566,
          3.2302,  4.7895],
        [-0.7844, -1.5343, -0.0742, -1.4279, -1.0936, -0.3646,  3.1682, -1.2314,
         -0.2716,  3.2576],
        [ 6.5356,  

# Verify Setup
Make sure cuda (GPU) is available

In [4]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [5]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [6]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [7]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [8]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [9]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [10]:
import os
os.chdir('/home/taekim/Garnella')

In [11]:
%load_ext autoreload
%autoreload 2


In [12]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [13]:
import torch
print("PyTorch CUDA version:", torch.version.cuda)
print("GPU visible:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

PyTorch CUDA version: 12.8
GPU visible: True
GPU name: NVIDIA GeForce RTX 5060 Ti


In [14]:
from embeddings import *
from models import *
from train_loop_caching import train_loop

# Cell 1: Embedding comparison (same model, different embeddings)
combinations_embeddings = [
    #(get_tfidf_embeddings, get_logistic_regression),
   # (get_multilingual_embeddings, get_logistic_regression),
    (get_gemma_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_logistic_regression),
]
results_emb = train_loop(train_df, val_df, combinations_embeddings)
pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)

/home/taekim/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Encoding:   0%|          | 0/3544 [00:01<?, ?it/s]


TypeError: Got unsupported ScalarType BFloat16

In [ ]:
# Cell 2: Model comparison (best embeddings, different classifiers)
combinations_models = [
    (get_qwen_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_linear_svm),
    (get_qwen_embeddings, get_mlp),
    (get_qwen_embeddings, get_xgboost),
    (get_qwen_embeddings, get_knn),
    (get_gemma_embeddings, get_mlp),
    (get_gemma_embeddings, get_xgboost),
]
results_models = train_loop(train_df, val_df, combinations_models)
pd.DataFrame(results_models).sort_values("validation_score", ascending=False)